# Facial-Emotion Recognition — Model Training
**Module:** Facial-Emotion Recognition

**Workflow:** Load dataset → detect faces / extract frames → preprocess → train CNN on 4 emotion categories (happy, sad, excited, neutral) → evaluate → convert to TensorFlow Lite for on-device deployment.

**Dataset:** Kaggle — `fahadullaha/facial-emotion-recognition-dataset`

Run cells top to bottom in Google Colab (Runtime → Change runtime type → GPU).

## 1. Setup and Kaggle authentication

In [ ]:
!pip install -q kaggle opencv-python-headless tensorflow scikit-learn matplotlib seaborn

import os, shutil, zipfile
from google.colab import files

print("Upload your kaggle.json (Kaggle account -> Create New API Token)")
uploaded = files.upload()  # upload kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d fahadullaha/facial-emotion-recognition-dataset -p /content/data --unzip

DATA_ROOT = '/content/data'
for root, dirs, fnames in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, '').count(os.sep)
    if depth < 2:
        print(root, '->', dirs)

## 2. Inspect classes and map to the 4 target categories
The raw dataset may ship with the standard FER labels (angry, disgust, fear, happy, neutral, sad, surprise).
We map those down to our four project categories: **happy, sad, excited, neutral**.

> Check the printed folder names above and adjust `CLASS_MAP` below to match exactly what you see.

In [ ]:
# Adjust the keys on the left to match the ACTUAL folder/class names printed above
CLASS_MAP = {
    'happy':    'happy',
    'sad':      'sad',
    'surprise': 'excited',   # surprise -> excited
    'neutral':  'neutral',
    # classes not needed for this project (uncomment/edit as required):
    # 'angry':  None,
    # 'disgust':None,
    # 'fear':   None,
}

TARGET_CLASSES = ['happy', 'sad', 'excited', 'neutral']
print('Target classes:', TARGET_CLASSES)

## 3. Face detection + frame extraction (OpenCV)

In [ ]:
import cv2
import numpy as np

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
IMG_SIZE = 96

def detect_and_crop_face(img_path, size=IMG_SIZE):
    """Load an image, detect the largest face, crop, resize and grayscale-normalize it.
    Falls back to the full image (resized) if no face is detected — common for
    already-cropped FER datasets."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        face = gray[y:y+h, x:x+w]
    else:
        face = gray
    face = cv2.resize(face, (size, size))
    face = face.astype('float32') / 255.0
    return face

## 4. Build the dataset (images -> arrays)

In [ ]:
X, y = [], []
label_to_idx = {c: i for i, c in enumerate(TARGET_CLASSES)}

# Adjust this path to wherever the extracted dataset's class folders live
# e.g. /content/data/train/<class_name>/*.jpg
SEARCH_ROOT = DATA_ROOT

for src_class, target_class in CLASS_MAP.items():
    if target_class is None:
        continue
    for root, dirs, fnames in os.walk(SEARCH_ROOT):
        if os.path.basename(root).lower() == src_class.lower():
            for fn in fnames:
                if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                    face = detect_and_crop_face(os.path.join(root, fn))
                    if face is not None:
                        X.append(face)
                        y.append(label_to_idx[target_class])

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)
print('Total samples:', X.shape[0])
print('Per-class counts:', {c: int((y == i).sum()) for c, i in label_to_idx.items()})

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, num_classes=len(TARGET_CLASSES))
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. CNN model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=4):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_cnn(num_classes=len(TARGET_CLASSES))
model.summary()

---
This notebook covers dataset collection, environment setup, preprocessing, and the baseline CNN architecture definition only.
Training, evaluation, and TensorFlow Lite conversion are Week 2 work and will be added in a separate commit.